In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

#motor parameters(healthy)
params = {
    'R'  : 1.0,
    'L'  : 0.01,
    'Ke' : 0.01,
    'Kt' : 0.01,
    'J'  : 0.01,
    'B'  : 0.001,
    'V'  : 12.0,
    'TL' : 0.05,
}

# Mild, repeatable diagnostic inputs. Every machine receives the same sequence.
# The multipliers are deliberately small so this is a diagnostic probe, not a stress test.
STIMULUS_PROFILE = {
    'load_torque': {'time': 2.0, 'multiplier': 1.05},   # +5% load torque
    'viscous_drag': {'time': 3.0, 'multiplier': 1.10}, # +10% controlled brake drag
    'voltage_dip': {'time': 4.0, 'multiplier': 0.97},  # -3% supply voltage
}

#physics functions
def step_multiplier(t, profile, name):
    """Return 1 before a scheduled stimulus and its multiplier afterwards."""
    step = profile.get(name)
    return 1.0 if step is None or t < step['time'] else step['multiplier']

def motor_derivatives(t, state, params):
    i     = state[0]
    omega = state[1]

    R  = params['R']
    L  = params['L']
    Ke = params['Ke']
    Kt = params['Kt']
    J  = params['J']
    profile = params.get('stimulus_profile', STIMULUS_PROFILE)
    B  = params['B'] * step_multiplier(t, profile, 'viscous_drag')
    V  = params['V'] * step_multiplier(t, profile, 'voltage_dip')
    TL = params['TL'] * step_multiplier(t, profile, 'load_torque')

    di_dt     = (V - R*i - Ke*omega) / L
    domega_dt = (Kt*i - B*omega - TL) / J

    return [di_dt, domega_dt]

def steady_state_initial_state(sim_params):
    """Healthy/fault-specific operating point before the diagnostic sequence begins."""
    R, B = sim_params['R'], sim_params['B']
    Ke, Kt = sim_params['Ke'], sim_params['Kt']
    omega_0 = (Kt * sim_params['V'] / R - sim_params['TL']) / (B + Kt * Ke / R)
    current_0 = (sim_params['V'] - Ke * omega_0) / R
    return [current_0, omega_0]

#simulation
t_span = (0, 5)
t_eval = np.linspace(0, 5, 5000)
initial_state = steady_state_initial_state(params)

solution = solve_ivp(
    fun=motor_derivatives,
    t_span=t_span,
    y0=initial_state,
    t_eval=t_eval,
    args=(params,),
    method='RK45',
    rtol=1e-6,
    atol=1e-8
)

time    = solution.t
current = solution.y[0]
speed   = solution.y[1]

#plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(time, current, color='royalblue', linewidth=1.5)
for name, step in STIMULUS_PROFILE.items():
    ax1.axvline(step['time'], color='gray', linestyle='--', linewidth=1, label=name.replace('_', ' '))
ax1.set_ylabel('Current (A)')
ax1.set_title('DC Motor Response to Controlled Multi-Parameter Test')
ax1.legend()
ax1.grid(True)

ax2.plot(time, speed, color='darkorange', linewidth=1.5)
for name, step in STIMULUS_PROFILE.items():
    ax2.axvline(step['time'], color='gray', linestyle='--', linewidth=1, label=name.replace('_', ' '))
ax2.set_ylabel('Speed (rad/s)')
ax2.set_xlabel('Time (s)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('../plots/healthy_motor_response.png', dpi=150)
plt.close()
print("Simulation complete. Plot saved to plots folder.")

Simulation complete. Plot saved to plots folder.


In [ ]:
# healthy motor comparison with unhealthy motor
def run_motor_simulation(sim_params, duration=5.0, n_samples=5000):
    
    result = solve_ivp(
        fun=motor_derivatives,
        t_span=(0, duration),
        y0=steady_state_initial_state(sim_params),
        t_eval=np.linspace(0, duration, n_samples),
        args=(sim_params,),
        method='RK45',
        rtol=1e-6,
        atol=1e-8
    )
    if not result.success:
        raise RuntimeError(f'Motor simulation failed: {result.message}')
    return result.t, result.y[0], result.y[1]

fault_parameters = {
    'Healthy': params.copy(),
    'Bearing fault (B x2.0)': {**params, 'B': params['B'] * 2.0},
    'Winding fault (R x1.5)': {**params, 'R': params['R'] * 1.5},
    'Mechanical fault (J x1.3)': {**params, 'J': params['J'] * 1.3},
    'Supply fault (V x0.90)': {**params, 'V': params['V'] * 0.90},
}

responses = {
    label: run_motor_simulation(sim_params)
    for label, sim_params in fault_parameters.items()
}

fig, (ax_current, ax_speed) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
colours = ['black', 'crimson', 'royalblue', 'darkorange', 'seagreen']
for (label, (sim_time, sim_current, sim_speed)), colour in zip(responses.items(), colours):
    ax_current.plot(sim_time, sim_current, label=label, color=colour, linewidth=1.4)
    ax_speed.plot(sim_time, sim_speed, label=label, color=colour, linewidth=1.4)

for axis in (ax_current, ax_speed):
    for name, step in STIMULUS_PROFILE.items():
        axis.axvline(step['time'], color='gray', linestyle='--', linewidth=1, label=name.replace('_', ' '))
    axis.grid(True, alpha=0.3)
    axis.legend(loc='best')

ax_current.set_title('Healthy and Degraded DC Motor Responses')
ax_current.set_ylabel('Current (A)')
ax_speed.set_ylabel('Speed (rad/s)')
ax_speed.set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('../plots/healthy_vs_degraded_responses.png', dpi=150, bbox_inches='tight')
plt.close()
print('Stage 2 complete. Saved plots/healthy_vs_degraded_responses.png')


In [ ]:
#extract the fingerprint
from scipy.stats import kurtosis

FEATURE_NAMES = [
    'rpm_drop_pct', 'settling_time_s', 'current_peak_a', 'current_rms_a',
    'current_rise_pct', 'speed_rms_rad_s', 'overshoot_pct',
    'speed_kurtosis', 'current_kurtosis', 'current_crest_factor'
]

def extract_fingerprint(time, current, speed, stimulus_time=2.0):
    
    time = np.asarray(time)
    current = np.asarray(current)
    speed = np.asarray(speed)
    post_mask = time >= stimulus_time
    if post_mask.sum() < 10:
        raise ValueError('At least ten post-stimulus samples are required.')

    t = time[post_mask]
    i = current[post_mask]
    w = speed[post_mask]
    i_initial, w_initial = i[0], w[0]
    tail_start = max(1, int(0.9 * len(t)))
    i_final = np.mean(i[tail_start:])
    w_final = np.mean(w[tail_start:])

    #positive rpm drop
    rpm_drop_pct = 100 * max(0.0, (w_initial - np.min(w)) / max(abs(w_initial), 1e-12))
    current_peak_a = np.max(np.abs(i))
    current_rms_a = np.sqrt(np.mean(i ** 2))
    current_rise_pct = 100 * max(0.0, (np.max(i) - i_initial) / max(abs(i_initial), 1e-12))
    speed_rms_rad_s = np.sqrt(np.mean(w ** 2))

    #overshoot measure
    speed_change = w_final - w_initial
    if abs(speed_change) < 1e-12:
        overshoot_pct = 0.0
        settling_time_s = 0.0
    else:
        direction = np.sign(speed_change)
        overshoot_pct = 100 * max(0.0, np.max(direction * (w - w_final)) / abs(speed_change))
        tolerance = 0.02 * abs(speed_change)
        outside_band = np.abs(w - w_final) > tolerance
        settling_index = np.where(outside_band)[0][-1] + 1 if np.any(outside_band) else 0
        settling_time_s = (t[min(settling_index, len(t) - 1)] - stimulus_time)

    fingerprint = {
        'rpm_drop_pct': rpm_drop_pct,
        'settling_time_s': settling_time_s,
        'current_peak_a': current_peak_a,
        'current_rms_a': current_rms_a,
        'current_rise_pct': current_rise_pct,
        'speed_rms_rad_s': speed_rms_rad_s,
        'overshoot_pct': overshoot_pct,
        'speed_kurtosis': kurtosis(w, fisher=False, bias=False),
        'current_kurtosis': kurtosis(i, fisher=False, bias=False),
        'current_crest_factor': current_peak_a / max(current_rms_a, 1e-12),
    }
    return fingerprint

# four stage 2 response types inspection part (under debugging)
fingerprints = {
    label: extract_fingerprint(sim_time, sim_current, sim_speed)
    for label, (sim_time, sim_current, sim_speed) in responses.items()
}
for label, fingerprint in fingerprints.items():
    print(f'\n{label}')
    for feature in FEATURE_NAMES:
        print(f'  {feature}: {fingerprint[feature]:.5f}')


In [ ]:
# Dataset configuration only — this cell defines classes and variation ranges; it does not generate data.
# Static parameter changes describe a fault. STIMULUS_PROFILE remains identical for every class.
DATASET_CLASS_SPECS = {
    'Healthy': {
        'fault_multiplier_ranges': {},
        'description': 'Normal motor with small manufacturing/operating variation.',
    },
    'Bearing fault': {
        'fault_multiplier_ranges': {'B': (1.30, 2.50)},
        'description': 'Higher permanent viscous friction from bearing wear.',
    },
    'Winding fault': {
        'fault_multiplier_ranges': {'R': (1.20, 2.00)},
        'description': 'Higher permanent winding resistance from damage or poor connection.',
    },
    'Mechanical fault': {
        'fault_multiplier_ranges': {'J': (1.10, 1.40)},
        'description': 'Higher permanent inertia from imbalance or added rotating mass.',
    },
    'Supply fault': {
        'fault_multiplier_ranges': {'V': (0.85, 0.95)},
        'description': 'Persistently reduced available supply voltage.',
    },
}

# Applied to every base parameter before the class-specific fault multiplier.
HEALTHY_VARIATION_RANGE = (0.95, 1.05)
NOISE_FRACTION_OF_STD = 0.02

print('Dataset setup ready. Classes:', ', '.join(DATASET_CLASS_SPECS))
print('Shared diagnostic sequence:', STIMULUS_PROFILE)


In [ ]:
# Stage 4 — generate a reproducible, physically bounded synthetic dataset.
# All classes receive the identical STIMULUS_PROFILE; only their motor properties differ.
import pandas as pd
from pathlib import Path

RANDOM_SEED = 20260918
SAMPLES_PER_CLASS = 12_000
DATASET_SIMULATION_SAMPLES = 1_001  # 5 ms sampling over the five-second experiment

# These are normal unit-to-unit / operating-condition variations, not faults.
NORMAL_PARAMETER_RANGES = {
    'R': (0.97, 1.03),
    'L': (0.95, 1.05),
    'J': (0.95, 1.05),
    'B': (0.90, 1.10),
    'V': (0.99, 1.01),
    'TL': (0.95, 1.05),
}
GAIN_RANGE = (0.98, 1.02)  # Ke and Kt use the same factor because they are physically coupled.

def sample_realistic_parameters(label, rng):
    """Sample one plausible motor, then apply only the requested permanent fault."""
    sampled = params.copy()
    for name, bounds in NORMAL_PARAMETER_RANGES.items():
        sampled[name] *= rng.uniform(*bounds)

    gain_factor = rng.uniform(*GAIN_RANGE)
    sampled['Ke'] *= gain_factor
    sampled['Kt'] *= gain_factor

    fault_multipliers = {}
    for name, bounds in DATASET_CLASS_SPECS[label]['fault_multiplier_ranges'].items():
        multiplier = rng.uniform(*bounds)
        sampled[name] *= multiplier
        fault_multipliers[name] = multiplier

    # TL above is only the nominal operating load. The scheduled test remains exactly +5% of it.
    return sampled, fault_multipliers

def simulate_motor_batch(parameter_arrays, duration=5.0, n_samples=DATASET_SIMULATION_SAMPLES):
    """Vectorised fixed-step RK4 integration of the same motor ODEs.

    This is used only for high-volume data generation; the notebook's reference
    simulator above remains solve_ivp/RK45. A 5 ms RK4 step resolves the fastest
    electrical time constant (about 10 ms) while making 60,000 runs practical.
    """
    t = np.linspace(0.0, duration, n_samples)
    n = len(parameter_arrays['R'])
    R, L = parameter_arrays['R'].to_numpy(), parameter_arrays['L'].to_numpy()
    Ke, Kt = parameter_arrays['Ke'].to_numpy(), parameter_arrays['Kt'].to_numpy()
    J, B = parameter_arrays['J'].to_numpy(), parameter_arrays['B'].to_numpy()
    V, TL = parameter_arrays['V'].to_numpy(), parameter_arrays['TL'].to_numpy()

    omega = (Kt * V / R - TL) / (B + Kt * Ke / R)
    current = (V - Ke * omega) / R
    current_history = np.empty((n, n_samples))
    speed_history = np.empty((n, n_samples))
    current_history[:, 0], speed_history[:, 0] = current, omega

    def derivatives(i, w, time_point):
        b_eff = B * step_multiplier(time_point, STIMULUS_PROFILE, 'viscous_drag')
        v_eff = V * step_multiplier(time_point, STIMULUS_PROFILE, 'voltage_dip')
        tl_eff = TL * step_multiplier(time_point, STIMULUS_PROFILE, 'load_torque')
        return (v_eff - R * i - Ke * w) / L, (Kt * i - b_eff * w - tl_eff) / J

    for index in range(n_samples - 1):
        dt = t[index + 1] - t[index]
        k1_i, k1_w = derivatives(current, omega, t[index])
        k2_i, k2_w = derivatives(current + 0.5 * dt * k1_i, omega + 0.5 * dt * k1_w, t[index] + 0.5 * dt)
        k3_i, k3_w = derivatives(current + 0.5 * dt * k2_i, omega + 0.5 * dt * k2_w, t[index] + 0.5 * dt)
        k4_i, k4_w = derivatives(current + dt * k3_i, omega + dt * k3_w, t[index + 1])
        current += dt * (k1_i + 2 * k2_i + 2 * k3_i + k4_i) / 6
        omega += dt * (k1_w + 2 * k2_w + 2 * k3_w + k4_w) / 6
        current_history[:, index + 1], speed_history[:, index + 1] = current, omega
    return t, current_history, speed_history

def extract_fingerprints_batch(time, current, speed):
    """Vectorised equivalent of extract_fingerprint for a batch of noisy signals."""
    post = time >= 2.0
    t, i, w = time[post], current[:, post], speed[:, post]
    i_initial, w_initial = i[:, 0], w[:, 0]
    tail_start = max(1, int(0.9 * len(t)))
    w_final = np.mean(w[:, tail_start:], axis=1)
    speed_change = w_final - w_initial
    safe_change = np.maximum(np.abs(speed_change), 1e-12)
    direction = np.sign(speed_change)
    outside_band = np.abs(w - w_final[:, None]) > (0.02 * safe_change)[:, None]
    has_outside = np.any(outside_band, axis=1)
    last_outside = len(t) - 1 - np.argmax(outside_band[:, ::-1], axis=1)
    settling_index = np.where(has_outside, np.minimum(last_outside + 1, len(t) - 1), 0)
    current_peak = np.max(np.abs(i), axis=1)
    current_rms = np.sqrt(np.mean(i ** 2, axis=1))
    return pd.DataFrame({
        'rpm_drop_pct': 100 * np.maximum(0.0, (w_initial - np.min(w, axis=1)) / np.maximum(np.abs(w_initial), 1e-12)),
        'settling_time_s': t[settling_index] - 2.0,
        'current_peak_a': current_peak,
        'current_rms_a': current_rms,
        'current_rise_pct': 100 * np.maximum(0.0, (np.max(i, axis=1) - i_initial) / np.maximum(np.abs(i_initial), 1e-12)),
        'speed_rms_rad_s': np.sqrt(np.mean(w ** 2, axis=1)),
        'overshoot_pct': 100 * np.maximum(0.0, np.max(direction[:, None] * (w - w_final[:, None]), axis=1) / safe_change),
        'speed_kurtosis': kurtosis(w, axis=1, fisher=False, bias=False),
        'current_kurtosis': kurtosis(i, axis=1, fisher=False, bias=False),
        'current_crest_factor': current_peak / np.maximum(current_rms, 1e-12),
    })

def generate_fingerprint_dataset(samples_per_class=SAMPLES_PER_CLASS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    feature_frames, audit_frames = [], []
    sample_id = 0
    total_samples = samples_per_class * len(DATASET_CLASS_SPECS)

    for label in DATASET_CLASS_SPECS:
        sampled_records, audit_records = [], []
        for offset in range(samples_per_class):
            sampled_params, fault_multipliers = sample_realistic_parameters(label, rng)
            sampled_records.append(sampled_params)
            audit_records.append({
                'sample_id': sample_id + offset, 'label': label, **sampled_params,
                'fault_multipliers': str(fault_multipliers),
            })

        parameter_frame = pd.DataFrame(sampled_records)
        sim_time, sim_current, sim_speed = simulate_motor_batch(parameter_frame)
        current_noise_fraction = rng.uniform(0.005, NOISE_FRACTION_OF_STD, size=samples_per_class)
        speed_noise_fraction = rng.uniform(0.005, NOISE_FRACTION_OF_STD, size=samples_per_class)
        sim_current += rng.normal(0, np.std(sim_current, axis=1)[:, None] * current_noise_fraction[:, None], sim_current.shape)
        sim_speed += rng.normal(0, np.std(sim_speed, axis=1)[:, None] * speed_noise_fraction[:, None], sim_speed.shape)

        class_features = extract_fingerprints_batch(sim_time, sim_current, sim_speed)
        if not np.all(np.isfinite(class_features[FEATURE_NAMES].to_numpy())):
            raise ValueError(f'Non-finite fingerprint in {label} samples.')
        class_features.insert(0, 'sample_id', np.arange(sample_id, sample_id + samples_per_class))
        class_features['label'] = label
        feature_frames.append(class_features)

        class_audit = pd.DataFrame(audit_records)
        class_audit['current_noise_fraction'] = current_noise_fraction
        class_audit['speed_noise_fraction'] = speed_noise_fraction
        audit_frames.append(class_audit)
        sample_id += samples_per_class
        print(f'Completed {label}: {sample_id:,}/{total_samples:,} samples')

    return pd.concat(feature_frames, ignore_index=True), pd.concat(audit_frames, ignore_index=True)

fingerprint_dataset, parameter_audit = generate_fingerprint_dataset()
output_dir = Path('../data')
output_dir.mkdir(parents=True, exist_ok=True)
fingerprint_dataset.to_csv(output_dir / 'fingerprint_dataset.csv', index=False)
parameter_audit.to_csv(output_dir / 'fingerprint_parameter_audit.csv', index=False)

print(f'\nSaved {len(fingerprint_dataset):,} ML-ready rows to data/fingerprint_dataset.csv')
print('Saved source-parameter audit trail to data/fingerprint_parameter_audit.csv')
print(fingerprint_dataset['label'].value_counts().sort_index())


In [ ]:
# Stage 5 — quality gate and healthy-baseline normalization.
# Rebuild the raw and ML-ready outputs after the extended settling-time correction.
import pandas as pd
raw_dataset = pd.read_csv('../data/fingerprint_dataset.csv')
parameter_audit = pd.read_csv('../data/fingerprint_parameter_audit.csv')

# The five-second response record is intentionally short for the transient features.
# For a physically valid 2% settling measure, propagate the exact linear motor
# dynamics after the final t=4 s voltage input for up to 60 additional seconds.
# This retains every sampled motor and noise realization while correcting the
# previously censored settling-time feature.
def propagate_linear_motor(i, w, R, L, Ke, Kt, J, B, V, TL, duration):
    a, b = -R / L, -Ke / L
    c, d = Kt / J, -B / J
    q_i, q_w = V / L, -TL / J
    determinant = a * d - b * c
    i_ss = (-d * q_i + b * q_w) / determinant
    w_ss = (c * q_i - a * q_w) / determinant
    trace_half = (a + d) / 2
    root = np.sqrt(((a - d) / 2) ** 2 + b * c)
    slow, fast = trace_half + root, trace_half - root
    denominator = slow - fast
    exp_slow, exp_fast = np.exp(slow * duration), np.exp(fast * duration)
    e11 = (exp_slow * (a - fast) - exp_fast * (a - slow)) / denominator
    e12 = b * (exp_slow - exp_fast) / denominator
    e21 = c * (exp_slow - exp_fast) / denominator
    e22 = (exp_slow * (d - fast) - exp_fast * (d - slow)) / denominator
    return i_ss + e11 * (i - i_ss) + e12 * (w - w_ss), w_ss + e21 * (i - i_ss) + e22 * (w - w_ss)

def calculate_extended_settling_time(audit, horizon_s=60.0, step_s=0.02):
    """Return 2%-band settling time measured from the first t=2 s input."""
    R, L = audit['R'].to_numpy(), audit['L'].to_numpy()
    Ke, Kt = audit['Ke'].to_numpy(), audit['Kt'].to_numpy()
    J, B = audit['J'].to_numpy(), audit['B'].to_numpy()
    V, TL = audit['V'].to_numpy(), audit['TL'].to_numpy()

    # State is at the steady operating point until the load step at t=2 s.
    i0 = (V - Ke * ((Kt * V / R - TL) / (B + Kt * Ke / R))) / R
    w0 = (Kt * V / R - TL) / (B + Kt * Ke / R)
    i3, w3 = propagate_linear_motor(i0, w0, R, L, Ke, Kt, J, B, V, TL * 1.05, 1.0)
    i4, w4 = propagate_linear_motor(i3, w3, R, L, Ke, Kt, J, B * 1.10, V, TL * 1.05, 1.0)

    # Final dynamics after the voltage dip; obtain its steady state and eigenmodes.
    final_v, final_b, final_tl = V * 0.97, B * 1.10, TL * 1.05
    a, b, c, d = -R / L, -Ke / L, Kt / J, -final_b / J
    determinant = a * d - b * c
    q_i, q_w = final_v / L, -final_tl / J
    w_final = (c * q_i - a * q_w) / determinant
    trace_half = (a + d) / 2
    root = np.sqrt(((a - d) / 2) ** 2 + b * c)
    slow, fast = trace_half + root, trace_half - root
    mode_denominator = slow - fast
    slow_coefficient = (c * (i4 - ( -d * q_i + b * q_w) / determinant) + (d - fast) * (w4 - w_final)) / mode_denominator
    fast_coefficient = (-c * (i4 - ( -d * q_i + b * q_w) / determinant) - (d - slow) * (w4 - w_final)) / mode_denominator
    tolerance = 0.02 * np.maximum(np.abs(w4 - w_final), 1e-9)
    timeline = np.arange(0.0, horizon_s + step_s, step_s)
    settling_after_final = np.empty(len(audit))

    for start in range(0, len(audit), 1_000):
        stop = min(start + 1_000, len(audit))
        error = np.abs(
            slow_coefficient[start:stop, None] * np.exp(slow[start:stop, None] * timeline)
            + fast_coefficient[start:stop, None] * np.exp(fast[start:stop, None] * timeline)
        )
        outside = error > tolerance[start:stop, None]
        last_outside = len(timeline) - 1 - np.argmax(outside[:, ::-1], axis=1)
        has_outside = np.any(outside, axis=1)
        settling_index = np.where(has_outside, np.minimum(last_outside + 1, len(timeline) - 1), 0)
        settling_after_final[start:stop] = timeline[settling_index]

    return 2.0 + settling_after_final

raw_dataset['settling_time_s'] = calculate_extended_settling_time(parameter_audit)
if raw_dataset['settling_time_s'].nunique() <= 1:
    raise ValueError('Extended settling calculation did not produce useful variation.')
raw_dataset.to_csv('../data/fingerprint_dataset.csv', index=False)

if raw_dataset.isna().any().any():
    raise ValueError('Dataset contains missing values.')
if raw_dataset['label'].value_counts().nunique() != 1:
    raise ValueError('Dataset classes are not balanced.')

constant_features = [feature for feature in FEATURE_NAMES if raw_dataset[feature].nunique() <= 1]
MODEL_FEATURE_NAMES = [feature for feature in FEATURE_NAMES if feature not in constant_features]
if not MODEL_FEATURE_NAMES:
    raise ValueError('No variable features remain for ML training.')

# Any constant feature is excluded automatically; extended settling time should now be retained.
healthy_rows = raw_dataset.loc[raw_dataset['label'] == 'Healthy', MODEL_FEATURE_NAMES]
baseline_mean = healthy_rows.mean()
baseline_std = healthy_rows.std(ddof=0)
if (baseline_std <= 0).any():
    raise ValueError('A retained feature has zero healthy-baseline standard deviation.')

ml_ready_dataset = raw_dataset[MODEL_FEATURE_NAMES + ['label']].copy()
normalized_dataset = (raw_dataset[MODEL_FEATURE_NAMES] - baseline_mean) / baseline_std
normalized_dataset['label'] = raw_dataset['label']
baseline_statistics = pd.DataFrame({
    'feature': MODEL_FEATURE_NAMES,
    'healthy_mean': baseline_mean[MODEL_FEATURE_NAMES].to_numpy(),
    'healthy_std': baseline_std[MODEL_FEATURE_NAMES].to_numpy(),
})

ml_ready_dataset.to_csv('../data/fingerprint_dataset_ml_ready.csv', index=False)
normalized_dataset.to_csv('../data/fingerprint_dataset_normalized.csv', index=False)
baseline_statistics.to_csv('../data/healthy_baseline_statistics.csv', index=False)

print(f'Raw rows: {len(raw_dataset):,}; balanced classes: {raw_dataset.label.value_counts().iloc[0]:,} each')
print('Excluded constant features:', constant_features or 'None')
print('ML features:', MODEL_FEATURE_NAMES)
print('Saved ML-ready, normalized, and healthy-baseline CSV files to data/.')
